# Computational Economics with SciPy
## A Data Analytics Project — Market Equilibrium, Optimization, Growth & Cycles

*Based on Steinkamp, V., "Python for Engineering and Scientific Computing" (2024),
Chapter 6 — "Numerical Computations and Simulations Using SciPy".*

Every numerical technique in this notebook (root-finding, constrained optimization,
interpolation, differentiation, integration, ODEs, FFT) mirrors an example from the
source chapter, retargeted from engineering signals to economic data. Work through
the notebook top to bottom; each section has:

1. **Theory** (markdown) — the economic model and the math behind it.
2. **Data** — a synthetic, reproducible dataset (fixed random seed).
3. **Task** — `TODO` cells for you to complete.
4. **Interpretation** — a short markdown cell where you explain the result in words.

Run the setup cell first.

## 0. Setup

Import the libraries used throughout the project. If `numdifftools` is not
installed, run `!pip install numdifftools --break-system-packages` in a cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import root, minimize
from scipy.interpolate import interp1d, CubicSpline
from scipy.integrate import quad, solve_ivp
from scipy.fft import fft, fftfreq
import numdifftools as nd

np.random.seed(42)          # reproducibility -- keep this seed unchanged
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

## 1. Market Equilibrium via Root-Finding
### (mirrors book §6.1, `scipy.optimize.root`)

**Theory.** A market is in equilibrium at the price `p*` where quantity demanded
equals quantity supplied. We use a *constant-elasticity* demand and supply system,
which — like the damped-oscillation example in the book — has no clean algebraic
solution, so we solve it numerically:

- Demand: `Qd(p) = A * p**(-eps_d)`  (eps_d = price elasticity of demand, eps_d > 0)
- Supply: `Qs(p) = B * p**(eps_s)`   (eps_s = price elasticity of supply, eps_s > 0)

Equilibrium condition: `f(p) = Qd(p) - Qs(p) = 0`.

Just as Listing 6.1 calls `root(f, x0, method='hybr')` to find the zeros of a
damped sine wave, we call `root()` on the excess-demand function `f(p)`.

In [ ]:
# ---- Synthetic market parameters (given) ----
A, eps_d = 500.0, 1.4     # demand scale and elasticity (elastic: eps_d > 1)
B, eps_s = 8.0, 1.2       # supply scale and elasticity

def demand(p):
    return A * p**(-eps_d)

def supply(p):
    return B * p**(eps_s)

In [ ]:
# TODO 1a: Define the excess-demand function f(p) = demand(p) - supply(p)
def excess_demand(p):
    ...  # your code here

# TODO 1b: Use scipy.optimize.root(excess_demand, p0, method='hybr')
#          starting from an initial guess p0 = [10.0]. Extract the
#          scalar equilibrium price into `p_star` and compute `q_star`.
p0 = [10.0]
p_star = None   # replace
q_star = None   # replace

print(f"Equilibrium price p* = {p_star}")
print(f"Equilibrium quantity q* = {q_star}")

# TODO 1c: Plot demand(p) and supply(p) over p_grid = np.linspace(1, 30, 300)
#          and mark the equilibrium point with ax.scatter(...).

**Interpretation (answer here).** What happens to the equilibrium price and
quantity if demand becomes *more* elastic (increase `eps_d`)? Try it and briefly
explain the economic intuition in 2–3 sentences.

## 2. Cost-Minimizing Input Mix (Constrained Optimization)
### (mirrors book §6.2, `scipy.optimize.minimize` — the cylinder-surface-area problem)

**Theory.** A firm produces output with a Cobb–Douglas technology
`Q(K, L) = K**alpha * L**beta` where `K` = capital, `L` = labor. Capital costs
`r` per unit and labor costs `w` per unit. The firm wants to produce a target
output `Q0` at minimum cost:

```
minimize   C(K, L) = r*K + w*L
subject to Q(K, L) = Q0,   K > 0, L > 0
```

This is exactly the structure of the tin-can (cylinder) minimization in the book
(minimize surface area subject to a fixed volume) — here we minimize cost subject
to a fixed output. We use `scipy.optimize.minimize` with an equality constraint
(SLSQP method).

In [ ]:
# ---- Given parameters ----
alpha, beta = 0.35, 0.65     # Cobb-Douglas exponents (decreasing/const/incr. returns -> alpha+beta)
r, w = 4.0, 2.5               # price of capital, price of labor
Q0 = 1000.0                   # required output level

In [ ]:
# TODO 2a: Define cost(x) = r*K + w*L where x = [K, L]
def cost(x):
    ...  # your code here

# TODO 2b: Define output_constraint(x) returning K**alpha * L**beta - Q0
def output_constraint(x):
    ...  # your code here

# TODO 2c: Call scipy.optimize.minimize(cost, x0, method="SLSQP",
#          bounds=bounds, constraints=[{"type": "eq", "fun": output_constraint}])
x0 = [50.0, 50.0]
bounds = [(1e-6, None), (1e-6, None)]
result = None  # replace

K_star, L_star = None, None  # replace with result.x
print(f"Cost-minimizing capital K* = {K_star}")
print(f"Cost-minimizing labor    L* = {L_star}")

# TODO 2d: Plot the isoquant Q=Q0 (solve L as a function of K) and the
#          isocost line through the optimum, and mark the optimal (K*, L*).

**Interpretation (answer here).** If the price of labor `w` doubles, would you
expect `K*` to rise or fall relative to `L*`? Re-run with `w = 5.0` and confirm.

## 3. Building a Yield Curve (Interpolation)
### (mirrors book §6.3, `scipy.interpolate.interp1d` — recovering a sampled signal)

**Theory.** Bond yields are only observed at a handful of standard maturities
(e.g., 3-month, 2-year, 10-year). To price a bond with a maturity that falls
*between* these points, or to compute the whole term structure, analysts
**interpolate**. Just as the book recovers a continuous signal from sampled
points, we recover a continuous yield curve from discrete market quotes.

We compare **linear interpolation** (`interp1d`, kind='linear') against a
**cubic spline** (`CubicSpline`), which is the standard choice for yield curves
because it is smooth (continuous first and second derivatives) — important
because the *slope* of the yield curve is itself economically meaningful
(term premium).

In [ ]:
# ---- Observed market yields at standard maturities (synthetic but realistic) ----
maturities = np.array([0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30])   # in years
yields_pct = np.array([4.85, 4.80, 4.55, 4.20, 4.05, 4.10, 4.25, 4.45, 4.70, 4.65])

df_yields = pd.DataFrame({"Maturity (yrs)": maturities, "Yield (%)": yields_pct})
df_yields

In [ ]:
# TODO 3a: Build a linear interpolant with interp1d(maturities, yields_pct, kind="linear")
lin_interp = None  # replace

# TODO 3b: Build a cubic spline interpolant with CubicSpline(maturities, yields_pct)
cubic_interp = None  # replace

# TODO 3c: Plot both curves over t_grid = np.linspace(0.25, 30, 400) together
#          with the original market quotes as scatter points.

# TODO 3d: Estimate the 4-year yield using both interpolants, and compute the
#          10y-2y term spread using the cubic spline (a classic yield-curve
#          recession indicator: a negative spread = "inverted curve").

**Interpretation (answer here).** Where do linear and cubic-spline estimates
differ the most, and why does that make sense given how the two methods work?
What does a negative 10y-2y spread mean economically?

## 4. Marginal Analysis & Market Surplus (Differentiation & Integration)
### (mirrors book §6.4 `numdifftools.Derivative` and §6.5 `scipy.integrate.quad`)

**Theory — differentiation.** A firm's total cost function is
`TC(Q) = 50 + 0.02*Q**2 + 4*Q`. The **marginal cost** is `MC(Q) = d(TC)/dQ`.
Just as the book differentiates a position function to get velocity
(free-fall example), we differentiate total cost to get marginal cost, using
`numdifftools.Derivative` exactly as in Listing 6.10.

**Theory — integration.** At the market equilibrium from Section 1
(`p*`, `q*`), **consumer surplus** is the area between the demand curve and the
horizontal line at `p*`, and **producer surplus** is the area between `p*` and
the supply curve:

```
CS = ∫₀^{q*} [ P_d(q) - p* ] dq
PS = ∫₀^{q*} [ p* - P_s(q) ] dq
```

where `P_d(q)` and `P_s(q)` are the **inverse** demand/supply functions
(price as a function of quantity). We compute these with `scipy.integrate.quad`,
exactly as the book integrates to find displacement or work done.

*Technical note:* for the constant-elasticity demand curve from Section 1, the
consumer-surplus integral only **converges** to a finite value when demand is
elastic (`eps_d > 1`) — that's why Section 1 uses `eps_d = 1.4`. If you set
`eps_d < 1`, try re-running this cell and see what `quad` reports; it's a nice
illustration of how a model's math (not just its economics) can break down.

In [ ]:
def TC(Q):
    return 50 + 0.02*Q**2 + 4*Q

# Inverse demand/supply (price as a function of quantity), consistent with
# Section 1's demand/supply if you like, but given directly here for clarity:
def P_demand(q):
    return (q / A) ** (-1/eps_d)     # inverse of demand(p) = A*p**-eps_d

def P_supply(q):
    return (q / B) ** (1/eps_s)      # inverse of supply(p) = B*p**eps_s

In [ ]:
# TODO 4a: Use numdifftools.Derivative(TC, n=1) to build MC(Q), evaluate it
#          over Q_grid = np.linspace(1, 100, 200), and plot it.
MC = None  # replace

# TODO 4b: Using scipy.integrate.quad, compute:
#          CS = integral of (P_demand(q) - p_star) dq from ~0 to q_star
#          PS = integral of (p_star - P_supply(q)) dq from ~0 to q_star
#          (p_star, q_star come from Section 1 -- rerun Section 1 first!)
CS = None  # replace
PS = None  # replace
print(f"Consumer surplus = {CS}")
print(f"Producer surplus = {PS}")

# TODO 4c: Recreate the surplus-area plot: inverse demand & supply curves,
#          a horizontal line at p_star, and shaded regions (ax.fill_between)
#          for consumer surplus and producer surplus.

**Interpretation (answer here).** Which is larger, consumer or producer
surplus, given the elasticities chosen in Section 1? Relate your answer to
the relative steepness (elasticity) of the two curves.

## 5. The Solow–Swan Growth Model (Differential Equations)
### (mirrors book §6.6, `scipy.integrate.solve_ivp` — spring-mass / population ODEs)

**Theory.** The Solow–Swan model describes how capital per worker `k(t)`
evolves over time:

```
dk/dt = s * k**alpha - (n + delta) * k
```

- `s`  = savings rate
- `alpha` = capital's share of output (Cobb-Douglas exponent)
- `n`  = population growth rate
- `delta` = capital depreciation rate

This is structurally identical to the second-order/first-order ODEs solved with
`solve_ivp` in the book (e.g., Listings 6.15–6.22) — here it's a first-order
nonlinear ODE with a stable fixed point (the steady state `k*`), the same kind
of long-run-equilibrium behavior as a damped oscillator settling to rest.

The steady state solves `s*k**alpha = (n+delta)*k`, giving
`k* = (s / (n+delta)) ** (1/(1-alpha))`.

In [ ]:
s_rate, alpha_sw, n_rate, delta_rate = 0.25, 0.33, 0.015, 0.06
k0 = 1.0          # initial capital per worker
t_span = (0, 100) # years
t_eval = np.linspace(*t_span, 400)

In [ ]:
# TODO 5a: Define solow(t, k) returning s_rate*k[0]**alpha_sw - (n_rate+delta_rate)*k[0]
def solow(t, k):
    ...  # your code here

# TODO 5b: Solve with solve_ivp(solow, t_span, [k0], t_eval=t_eval)
sol = None  # replace

# TODO 5c: Compute the analytical steady state
#          k_star = (s_rate / (n_rate + delta_rate)) ** (1/(1 - alpha_sw))
k_star = None  # replace

# TODO 5d: Plot k(t) from the simulation together with a horizontal dashed
#          line at k_star.

# TODO 5e (bonus): Compute output per worker y(t) = k(t)**alpha_sw and its
#          growth rate using np.gradient(y_t, sol.t) / y_t, and plot it.
#          What happens to the growth rate as k(t) -> k_star?

**Interpretation (answer here).** Increase the savings rate `s_rate` to 0.35
and re-run. Does the steady state `k*` rise or fall, and does the economy
temporarily grow faster or slower while transitioning to the new steady state?
This is the "Solow prediction" about savings-driven growth — summarize it in
2–3 sentences.

## 6. Detecting the Business Cycle (Fourier Transform)
### (mirrors book §6.7 and the bearing-vibration example in §6.9 —
### `scipy.fft.fft`/`fftfreq` used to find a hidden periodicity in a noisy signal)

**Theory.** Real GDP growth is noisy quarter to quarter, but many economies
exhibit a recurring **business cycle** (expansion/recession) with a period of
roughly 6–10 years. In the book, the exact same FFT technique is used to find
a hidden mechanical fault frequency buried in noisy vibration data (Listing
6.32, "Simulation of a Bearing Damage"). We apply it here to a synthetic GDP
growth series: trend + business-cycle oscillation + quarter-to-quarter noise.

In [ ]:
# ---- Synthetic quarterly GDP growth (%) over 60 years = 240 quarters ----
n_quarters = 240
t_q = np.arange(n_quarters)                  # quarter index
Ta = 0.25                                    # sampling interval in years (quarterly)

cycle_period_years = 8.0                     # the "true" business-cycle length we hid in the data
trend = 2.0                                  # average growth rate (%)
cycle_amplitude = 1.2
noise = np.random.normal(0, 0.5, n_quarters)

gdp_growth = (trend
              + cycle_amplitude * np.sin(2*np.pi * t_q * Ta / cycle_period_years)
              + noise)

fig, ax = plt.subplots()
ax.plot(t_q * Ta, gdp_growth)
ax.set(xlabel="Year", ylabel="GDP growth (%)", title="Synthetic Quarterly GDP Growth")
plt.show()

In [ ]:
# TODO 6a: De-mean the series: gdp_detrended = gdp_growth - gdp_growth.mean()
gdp_detrended = None  # replace

# TODO 6b: Compute the FFT with fft(gdp_detrended) and the frequency axis
#          with fftfreq(N, Ta) where N=n_quarters and Ta=0.25 (years/quarter).
G_fft = None   # replace
freqs = None   # replace

# TODO 6c: Compute normalized amplitudes amps = 2*np.abs(G_fft)/N, keep only
#          positive frequencies, and plot amplitude vs. frequency.

# TODO 6d: Find the frequency with the largest amplitude (excluding freq=0),
#          convert it to a period (1/frequency), and compare it to the
#          true hidden cycle length (cycle_period_years = 8.0).
dominant_freq = None    # replace
detected_period = None  # replace
print(f"Detected business-cycle length: {detected_period} years")

**Interpretation (answer here).** Increase the noise standard deviation
(e.g., to 1.5) and re-run. Is the FFT still able to recover the correct
cycle length? What does this tell you about using frequency-domain methods
on noisy macroeconomic data in practice?

## 7. Bonus: Technology Adoption — the Bass Diffusion Model
### (structurally the same ODE as the book's epidemic/SIR simulation, §6.11.5,
### Listing 6.34 — "Simulation of an Epidemic")

**Theory.** The Bass model describes how a new product or technology spreads
through a population of potential adopters — mathematically the same
"S-shaped diffusion" structure as an epidemic infecting a population:

```
dF/dt = (p + q*F) * (1 - F)
```

- `F(t)` = cumulative fraction of the population that has adopted by time `t`
- `p` = coefficient of innovation (adoption from advertising/external influence)
- `q` = coefficient of imitation (adoption from word-of-mouth/social influence,
  the same "contagion" mechanism as an epidemic's infection term)

We solve this with `solve_ivp`, exactly as the book solves the epidemic
`dgl(t, ya)` system, and plot the adoption curve and the "adoption rate"
(the new-adopters-per-period curve, which is the classic bell-shaped Bass curve).

In [ ]:
p_innov, q_imit = 0.03, 0.38     # typical consumer-durable estimates
F0 = 0.001                        # tiny initial fraction of early adopters
t_span_bass = (0, 20)             # years
t_eval_bass = np.linspace(*t_span_bass, 400)

In [ ]:
# TODO 7a: Define bass(t, F) returning (p_innov + q_imit*F[0]) * (1 - F[0])
def bass(t, F):
    ...  # your code here

# TODO 7b: Solve with solve_ivp(bass, t_span_bass, [F0], t_eval=t_eval_bass)
sol_bass = None  # replace

# TODO 7c: Compute the adoption rate dF/dt with np.gradient(F_t, sol_bass.t)
#          and plot BOTH the cumulative adoption curve F(t) and the
#          adoption-rate curve (side by side, like a product-launch S-curve
#          and its bell-shaped derivative).

# TODO 7d: Find the year at which the adoption rate peaks
#          (np.argmax on the adoption rate array).
t_peak = None  # replace
print(f"Peak adoption rate occurs at year {t_peak}")

**Interpretation (answer here).** Increase `q_imit` (word-of-mouth strength)
and re-run. What happens to the timing and height of the adoption peak? Which
real-world products would you expect to have high `q` vs. high `p`?

## 8. Wrap-Up

Write a short (150–250 word) summary in this cell covering:

- One insight from the market-equilibrium/optimization sections (micro).
- One insight from the growth/business-cycle sections (macro).
- Which SciPy technique from the book's Chapter 6 you found most directly
  transferable to economic data analytics, and why.

*(Your answer here.)*